# EDA and data preparation

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

def find_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'database.xlsx').exists():
            return candidate
    raise FileNotFoundError('Could not locate database.xlsx in the current directory or its parents.')

ROOT = find_project_root()
OUT = ROOT / 'outputs' / 'eda'
OUT.mkdir(parents=True, exist_ok=True)

DATA_PATH = ROOT / 'database.xlsx'
OUTPUT = OUT


In [ ]:
WORK_DIR

In [ ]:
DATABASE_CATEGORICAL = ['AM','Pr','Sp','Sp2','PM']
DATABASE_NUMERIC = ['AMc','Prc','Spc','Sp2c','MSP','CT','Ct','RT','RP','Rt','RH','T','P','GHSV','Inert','CO','CH4','H2O','H2/CO2']
MODEL_FEATURES = ['AM','AMc','Pr','Prc','Sp','Sp2','MSP','PM','CT','Ct','RT','Rt','RH','T','P','GHSV','Inert','H2/CO2']
MODEL_CATEGORICAL = ['AM','Pr','Sp','Sp2','PM']
MODEL_NUMERIC = [x for x in MODEL_FEATURES if x not in MODEL_CATEGORICAL]
TARGET = 'Rate'
FEATURE_GROUPS = {'Catalyst composition':['AM','AMc','Pr','Prc','Sp','Sp2','MSP'],'Preparation':['PM','CT','Ct','RT','Rt','RH'],'Operating condition':['T','P','GHSV','Inert','H2/CO2']}


In [ ]:
data = pd.read_excel(DATA_PATH)
required = ['Number of data','Reference','Catalyst',*DATABASE_CATEGORICAL,*DATABASE_NUMERIC,TARGET]
missing = sorted(set(required) - set(data.columns))
if missing: raise ValueError(f'Missing columns: {missing}')
if 'MSP' not in data.columns: raise ValueError('The public submission database must use MSP.')
print('shape:', data.shape)
print('duplicate rows:', int(data.duplicated().sum()))
print('target missing:', int(data[TARGET].isna().sum()))
data.info()


In [ ]:
missing_table = data[DATABASE_CATEGORICAL + DATABASE_NUMERIC + [TARGET]].isna().sum().rename('missing').to_frame()
missing_table.to_csv(OUTPUT / 'missing_values.csv')
display(missing_table)
display(data[DATABASE_NUMERIC + [TARGET]].describe().T)
data[DATABASE_NUMERIC + [TARGET]].describe().T.to_csv(OUTPUT / 'numeric_descriptive_statistics.csv')


In [ ]:
category_rows = []
for feature in DATABASE_CATEGORICAL:
    counts = data[feature].astype('string').fillna('Missing').value_counts(dropna=False)
    category_rows.append(pd.DataFrame({'feature':feature,'category':counts.index.astype(str),'count':counts.values,'fraction':counts.values / len(data)}))
category_table = pd.concat(category_rows, ignore_index=True)
category_table.to_csv(OUTPUT / 'categorical_frequencies.csv', index=False)
display(category_table)


In [ ]:
schema = pd.DataFrame({'column':data.columns,'dtype':data.dtypes.astype(str),'missing':data.isna().sum().values})
model_audit = schema.set_index('column').loc[MODEL_FEATURES + [TARGET]].reset_index()
model_audit.to_csv(OUTPUT / 'model_input_audit.csv', index=False)
print(f'all descriptors: {len(DATABASE_CATEGORICAL) + len(DATABASE_NUMERIC)}')
print(f'model predictors: {len(MODEL_FEATURES)} ({len(MODEL_NUMERIC)} numeric, {len(MODEL_CATEGORICAL)} categorical)')
display(model_audit)
